In [1]:
!nvidia-smi

Wed Oct  7 22:17:24 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   48C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
import numpy as np
import cupy as cp
from numba import cuda

print("NumPy", np.__version__, "| CuPy", cp.__version__)
print("Numba sees a GPU:", cuda.is_available())
print("GPU:", cuda.get_current_device().name)

NumPy 2.1.3 | CuPy 14.0.1
Numba sees a GPU: True
GPU: Tesla T4


In [3]:
import time

rng = np.random.default_rng(seed=0)

def make_matrices(n):
    A = rng.random((n, n), dtype=np.float32)
    B = rng.random((n, n), dtype=np.float32)
    return A, B

In [4]:
#matrix multiplication (the 3 loops show how this becomes n^3)
def matmul_python(A, B):
    n = len(A)
    C = [[0.0] * n for _ in range(n)]
    for i in range(n):
        for j in range(n):
            total = 0.0
            for k in range(n):
                total += A[i][k] * B[k][j]
            C[i][j] = total
    return C

In [5]:
A, B = make_matrices(64)
C_python = matmul_python(A.tolist(), B.tolist())
C_numpy = A @ B
print("Matches NumPy:", np.allclose(C_python, C_numpy, rtol=1e-4))

Matches NumPy: True


In [6]:
#doubling n should take about 8x longer as the loop is O(n^3)
for n in [32, 64, 128, 256]:
    A, B = make_matrices(n)
    A_list, B_list = A.tolist(), B.tolist()
    start = time.perf_counter()
    matmul_python(A_list, B_list)
    elapsed = time.perf_counter() - start
    print(f"n = {n:4}: {elapsed:.3f} seconds")


n =   32: 0.002 seconds
n =   64: 0.018 seconds
n =  128: 0.135 seconds
n =  256: 1.762 seconds


In [7]:
#NumPy (CPU, optimized BLAS)
for n in [32, 64, 128, 256, 512, 1024, 2048, 4096]:
    A, B = make_matrices(n)
    start = time.perf_counter()
    A @ B
    elapsed = time.perf_counter() - start
    print(f"n = {n:4}: {elapsed:.6f} seconds")

n =   32: 0.000231 seconds
n =   64: 0.000028 seconds
n =  128: 0.003251 seconds
n =  256: 0.000597 seconds
n =  512: 0.003331 seconds
n = 1024: 0.028991 seconds
n = 2048: 0.182182 seconds
n = 4096: 1.899289 seconds


In [8]:
# comparing the speed of NumPy and python
A, B = make_matrices(256)
A_list, B_list = A.tolist(), B.tolist()

start = time.perf_counter()
matmul_python(A_list, B_list)
python_time = time.perf_counter() - start

start = time.perf_counter()
A @ B
numpy_time = time.perf_counter() - start

print(f"Python: {python_time:.3f} s | NumPy: {numpy_time:.6f} s | Speedup: {python_time / numpy_time:,.0f}x")

Python: 2.597 s | NumPy: 0.000757 s | Speedup: 3,432x


In [9]:
# using CuPy instead by copying matrices to GPU memory to multiply,
# then copying the result back to check it
A, B = make_matrices(512)
A_gpu = cp.asarray(A)
B_gpu = cp.asarray(B)
C_gpu = A_gpu @ B_gpu
print("Matches NumPy:", np.allclose(cp.asnumpy(C_gpu), A @ B, rtol=1e-4))

Matches NumPy: True


In [10]:
# GPU calls return immediately, so timing without synchronize is wrong
A, B = make_matrices(4096)
A_gpu, B_gpu = cp.asarray(A), cp.asarray(B)
A_gpu @ B_gpu
cp.cuda.Device().synchronize()

start = time.perf_counter()
A_gpu @ B_gpu
no_sync = time.perf_counter() - start

start = time.perf_counter()
A_gpu @ B_gpu
cp.cuda.Device().synchronize()
with_sync = time.perf_counter() - start

print(f"Without synchronize: {no_sync:.6f} s  wrong b/c only measures sending the job")
print(f"With synchronize:    {with_sync:.6f} s  correct timing")

Without synchronize: 0.000463 s  (wrong: only measures sending the job)
With synchronize:    0.097610 s  (correct)


In [11]:
# The first GPU call includes one-time setup, so it's much slower than later calls
A, B = make_matrices(1024)
A_gpu, B_gpu = cp.asarray(A), cp.asarray(B)
for attempt in range(1, 4):
    start = time.perf_counter()
    A_gpu @ B_gpu
    cp.cuda.Device().synchronize()
    print(f"Call {attempt}: {time.perf_counter() - start:.6f} s")

Call 1: 0.001192 s
Call 2: 0.000988 s
Call 3: 0.000939 s


In [12]:
# GPU compute time only (data already on the GPU), with warm-up and synchronize
for n in [32, 64, 128, 256, 512, 1024, 2048, 4096]:
    A, B = make_matrices(n)
    A_gpu, B_gpu = cp.asarray(A), cp.asarray(B)
    A_gpu @ B_gpu                      # warm-up for this size
    cp.cuda.Device().synchronize()
    start = time.perf_counter()
    A_gpu @ B_gpu
    cp.cuda.Device().synchronize()
    elapsed = time.perf_counter() - start
    print(f"n = {n:4}: {elapsed:.6f} seconds")

n =   32: 0.000109 seconds
n =   64: 0.000092 seconds
n =  128: 0.000099 seconds
n =  256: 0.000164 seconds
n =  512: 0.000227 seconds
n = 1024: 0.000943 seconds
n = 2048: 0.006203 seconds
n = 4096: 0.047788 seconds
